# Entropy generator: nozzles with losses — replicating De Domenico et al. (2019)

This notebook validates the **Nefes perturbation network** against

> F. De Domenico, E. O. Rolland, S. Hochgreb, *A generalised model for acoustic
> and entropic transfer function of nozzles with losses*, **J. Sound Vib. 440
> (2019) 212–230**

A nozzle/orifice is described via four stations (their Fig. 3),

$$A_1 \;\xrightarrow{\text{isentropic}}\; A_T \;\xrightarrow{\text{isentropic}}\; A_j \;\xrightarrow{\text{Borda (momentum)}}\; A_2,$$

In fact, all practical subsonic orifice type flows can be expressed through this description.
The flow is isentropic up to a notional area $A_j$ in the divergent, then a momentum-conserving (Borda–Carnot) jump $A_j\!\to\!A_2$ for the loss.
The single loss parameter is $\beta = A_j/A_2$:

* $\beta = \beta_{\min} = A_T/A_2$  → **orifice plate, sudden expansion** (jet, maximum loss),
* $\beta = 1$  → **isentropic nozzle** (full pressure recovery, no loss),
* in between → a **non-isentropic nozzle**.

We build these from just two Nefes elements — `isentropic_area_change` and
`sudden_area_change` (whose momentum law *is* their Eq. 5/6) — and check, at the
**Cambridge Entropy Generator** geometry ($D_1=42.6$ mm, $D_T=6.6$ mm):

1. **Mean flow** — the upstream static-pressure rise $p_1/p_2$ vs Mach (their Fig. 5/9).
2. **Analytical anchor** — the assembled compact scattering matrix in the De Domenico
   $(P^+,P^-,\sigma)$ flavour (our `riemann` basis, their Eqs. 9–11) against an
   *independent* composition of the analytic jumps.
3. **Transfer functions** — $R^\pm, T^\pm$ and the entropy→sound coefficients vs throat
   Mach (their Fig. 6).
4. **Reflectivity & transmissivity** — the orifice-plate $R^+, T^+$ vs throat Mach
   (their Fig. 12), Nefes against the independent jump composition.
5. **Open-jet termination** — the reflected direct ($R$) and indirect ($S_R$) noise of a
   convergent nozzle/orifice discharging into the open (their Fig. 16 / Fig. 7).


In [ ]:
import os
import sys

# Add the repo root (the dir containing the `nefes` package) to the python path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nefes.elements import catalog as cat
from nefes.shell import build_problem
from nefes.thermo.configure import perfect_gas
from nefes.solver import solve
from nefes.solver.report import states_table
from nefes.assembly.recover import ES_RHO, ES_U, ES_P, ES_C, ES_M, ES_PT
from nefes.perturbation import perturbation_response
from nefes.perturbation.operator.characteristics import char_to_dq
from nefes.plotting import use_nefes_theme, COLORWAY

# Below is because plotly LaTeX rendering is not working out of box in notebooks viewed in VSCode/Cursor.
from IPython.display import display, HTML
import plotly.offline as pyo
pyo.init_notebook_mode()
display(HTML(
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-AMS-MML_SVG"></script>'
))

use_nefes_theme()

R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
K = CP / R
CFG = perfect_gas(R, GAMMA)
FULL = ("acoustic", "entropy")

# Cambridge Entropy Generator geometry (circular ducts)
D1, DT = 0.0426, 0.0066
A1 = A2 = np.pi / 4 * D1 ** 2
AT = np.pi / 4 * DT ** 2

## 1. Mean flow — upstream pressure rise (Fig. 5 and Fig. 9)

This demonstrates the total pressure loss associated with non-isentropic nozzle, and its conservation for the case of an isentropic nozzle.
For the **orifice plate** ($\beta=\beta_{\min}$) the flow contracts isentropically to the throat and then expands as a jet with a momentum-conserving (lossy) jump.
We sweep the inlet total pressure, solve the steady mean flow, and read the throat Mach and the upstream/downstream static-pressure ratio $p_1/p_2$.
The loss shows up as $p_1/p_2 > 1$ (upstream static pressure is higher), growing with Mach. 
The fully isentropic nozzle would recover all of it ($p_1/p_2 = 1$ for $A_1=A_2$).

In [ ]:
# Build a helper function that takes in parameters and returns the compiled problem, solution, and states table
def solve_orifice(pt_in):
    #           A1                        AT                  A2
    # reservoir -> isentropic contraction -> sudden expansion -> pressure outlet
    net = [cat.total_pressure_inlet(pt_in, 300.0), cat.isentropic_area_change(),
           cat.sudden_area_change(), cat.pressure_outlet(101325.0, 300.0)]
    prob = build_problem(CFG, net, [(0, 1, A1), (1, 2, AT), (2, 3, A2)], 1.0, 101325.0, CP * 300.0)
    sol = solve(prob)
    assert sol.converged
    return prob, sol, states_table(prob, sol.x)

# Sweep the inlet total pressure from just above the back pressure
n_pts = np.linspace(101400.0, 182000.0, 36)
M1, MT, p_ratio = [], [], []
for pt in n_pts:
    _, res, est = solve_orifice(pt)
    if not res.converged or est[ES_M, 1] >= 1.0:
        continue
    M1.append(est[ES_M, 0]); MT.append(est[ES_M, 1]); p_ratio.append(est[ES_P, 0] / est[ES_P, 2])
M1, MT, p_ratio = np.array(M1), np.array(MT), np.array(p_ratio)

fig = go.Figure()
fig.add_trace(go.Scatter(x=M1, y=p_ratio, mode="lines+markers", name="orifice (Nefes)", line_color=COLORWAY[0]))
fig.add_hline(y=1.0, line=dict(dash="dash", color=COLORWAY[4]),
              annotation_text="isentropic nozzle (no loss)", annotation_position="bottom right")
fig.update_layout(title="Upstream static-pressure rise across the orifice (Fig. 5)",
                  xaxis_title=r"$M_1$", yaxis_title=r"$p_1/p_2$", height=440, width=720)
fig.show()
print(f"throat Mach span: {MT.min():.2f} .. {MT.max():.2f}   (subsonic; M_T = 1 is the choke / v1 limit)")

## 2. Analytical anchor — compact scattering vs an independent jump composition

The De Domenico model solves a 9-unknown linear system $\mathbf{M}\,\mathbf{X} =
\mathbf{a}_1 P^+_1 + \mathbf{a}_2 P^-_2 + \mathbf{a}_3 \sigma_1$ (their Eq. 19): a
**scattering matrix** in the $(P^+,P^-,\sigma)$ flavour.  We reproduce it
*independently* by composing the analytic compact jumps (mass + total enthalpy +
[$p_t$ | momentum]), linearised by complex step — sharing no code with the Nefes
kernels — and compare to the assembled Nefes compact matrix at a representative
operating point.

In [ ]:
# --- independent analytic compact jumps (primitive vars), complex-step safe ---
def _derived(y):
    rho, u, p = y
    ht = K * p / rho + 0.5 * u * u
    M = u / (GAMMA * p / rho) ** 0.5
    return ht, p * (1.0 + 0.5 * (GAMMA - 1.0) * M * M) ** (GAMMA / (GAMMA - 1.0))

def r_isen(y0, y1, A0, A1):
    rho0, u0, p0 = y0; rho1, u1, p1 = y1
    ht0, pt0 = _derived(y0); ht1, pt1 = _derived(y1)
    return np.array([rho1*u1*A1 - rho0*u0*A0, ht1 - ht0, pt0 - pt1])

# De Domenico Eq. 5/6 (small-port pressure on the step)
def r_borda(y0, y1, A0, A1):  
    rho0, u0, p0 = y0; rho1, u1, p1 = y1
    ht0, _ = _derived(y0); ht1, _ = _derived(y1)
    md0, md1 = rho0*u0*A0, rho1*u1*A1
    ps = p0 if A0 <= A1 else p1
    return np.array([md1 - md0, ht1 - ht0, (md1*u1 + p1*A1) - (md0*u0 + p0*A0) - ps*(A1 - A0)])

def jump_tm(rf, y0, y1, *a):
    # Compute the linearized transfer map between primitive variable jumps at an interface
    # using complex step differentiation for Jacobians w.r.t y0 and y1.
    y0 = np.asarray(y0, complex)
    y1 = np.asarray(y1, complex)
    h = 1e-30  # Complex step size (very small)
    Ja = np.zeros((3, 3))
    Jb = np.zeros((3, 3))
    for k in range(3):
        # Partial derivatives w.r.t. y0 (holding y1 fixed)
        yp = y0.copy()
        yp[k] += 1j * h
        Ja[:, k] = rf(yp, y1, *a).imag / h
        # Partial derivatives w.r.t. y1 (holding y0 fixed)
        yp = y1.copy()
        yp[k] += 1j * h
        Jb[:, k] = rf(y0, yp, *a).imag / h
    # Solve for the transfer matrix: -Jb^{-1} * Ja
    return -np.linalg.solve(Jb, Ja)

prim = lambda est, e: np.array([est[ES_RHO, e], est[ES_U, e], est[ES_P, e]])

prob, sol, est = solve_orifice(140000.0)               # M_T ~ 0.71
T_char = perturbation_response(prob, sol.x, np.array([0.0]), excite=FULL).transfer_matrix(0, 2)[0]
R0 = char_to_dq(est[ES_RHO, 0], est[ES_C, 0]); R2 = char_to_dq(est[ES_RHO, 2], est[ES_C, 2])
T_fns_prim = (R2 @ T_char @ np.linalg.inv(R0)).real
T_ref_prim = (jump_tm(r_borda, prim(est, 1), prim(est, 2), AT, A2)
              @ jump_tm(r_isen, prim(est, 0), prim(est, 1), A1, AT)).real

rel_err = np.max(np.abs(T_fns_prim - T_ref_prim)) / np.max(np.abs(T_ref_prim))
print(f"Maximum relative error between Nefes and independent formulation shown above = {rel_err:.2e}")

## 3. Transfer functions vs throat Mach (Fig. 6)

Sweeping the operating point gives the compact acoustic ($R^\pm,T^\pm$) and entropic
transfer functions of the orifice plate versus throat Mach.  The **entropy→sound**
coefficients (the *indirect noise*) grow with Mach — the conversion an acoustics-only
model cannot capture — and the upstream acoustic reflection $R^+$ changes sign at low
Mach, the orifice cancellation discussed in their §3.

In [ ]:
def dedomenico_coeffs(pt_in):
    prob, res, est = solve_orifice(pt_in)
    if not res.converged or est[ES_M, 1] >= 1.0:
        return None
    S = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0].real
    p1, p2 = est[ES_P, 0], est[ES_P, 2]
    return est[ES_M, 1], dict(Rp=S[0, 0], Rm=S[1, 2], Tp=S[1, 0]*(p2/p1), Tm=S[0, 2]*(p1/p2),
                              SR=S[0, 1], ST=S[1, 1])

# From just above the back pressure (near-quiescent throat) up to the choke
n_pts = 48
# Quadratic clustering toward the lower value (101400.0)
quad = np.linspace(0, 1, n_pts) ** 2
pt_grid = 101400.0 + (182000.0 - 101400.0) * quad
rows = [r for r in (dedomenico_coeffs(pt) for pt in pt_grid) if r]
m = np.array([r[0] for r in rows])
C = {k: np.array([r[1][k] for r in rows]) for k in rows[0][1]}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Acoustic transfer functions", "Entropy → sound (indirect noise)"),
                    horizontal_spacing=0.12)
for k, lab, c in [("Rp", r"$R^+$", 0), ("Rm", r"$R^-$", 1), ("Tp", r"$T^+$", 2), ("Tm", r"$T^-$", 3)]:
    fig.add_trace(go.Scatter(x=m, y=C[k], name=lab, line_color=COLORWAY[c]), row=1, col=1)
for k, lab, c in [("SR", r"$\sigma \to \text{up}$", 5), ("ST", r"$\sigma \to \text{down}$", 6)]:
    fig.add_trace(go.Scatter(x=m, y=C[k], name=lab, line_color=COLORWAY[c]), row=1, col=2)

fig.add_hline(y=0.0, line=dict(dash="dot", color="#9aa5b1"), row=1, col=1)
fig.update_xaxes(title_text=r"$M_T$", row=1, col=1)
fig.update_xaxes(title_text=r"$M_T$", row=1, col=2)

LEGEND_POSITION = dict(
    orientation="h",
    yanchor="middle",
    y=1.11,
    yref="paper",
    xanchor="center",
    x=0.5,
    borderwidth=0,
)
fig.update_layout(
    title=dict(
        text="Compact transfer functions of the Cambridge orifice plate (Fig. 6)",
        x=0.5,
        xanchor="center",
        y=0.99,
        yanchor="top",
    ),
    height=460,
    width=980,
    margin=dict(t=75),
    legend=LEGEND_POSITION,
)
fig.show()

## 4. Orifice-plate reflectivity & transmissivity (Fig. 12)

For the orifice plate ($\beta=\beta_{\min}$) the compact reflection and transmission of a
downstream-running acoustic wave are (their Eq. 16)

$$R^+ = \frac{P^-_1}{P^+_1}, \qquad T^+ = \frac{P^+_2}{P^+_1}\,\frac{\gamma p_2}{\gamma p_1},$$

read from the assembled $(P^+,P^-,\sigma)$ scattering matrix. We sweep the throat
Mach $M_T$ and overlay the **independent** prediction obtained by composing the analytic compact
jumps of Sec. 2, reproducing the analytical curves of their Fig. 12 (black $R^+$, red $T^+$).

In [ ]:
# Convert an independent char transfer matrix (w_b = T w_a, w = (f,g,h)) into the
# (P+, P-, sigma) `riemann` scattering matrix: incoming [P+_1, sigma_1, P-_2].
def char_tm_to_riemann_sm(T, est, a, b):
    g = np.array([-T[1, 0] / T[1, 1], 1.0 / T[1, 1], -T[1, 2] / T[1, 1]])
    fb = T[0, 0] * np.array([1., 0, 0]) + T[0, 1] * g + T[0, 2] * np.array([0, 0, 1.])
    hb = T[2, 0] * np.array([1., 0, 0]) + T[2, 1] * g + T[2, 2] * np.array([0, 0, 1.])
    Sc = np.vstack([g, fb, hb])
    ca, cb, ra, rb = est[ES_C, a], est[ES_C, b], est[ES_RHO, a], est[ES_RHO, b]
    din = np.array([1 / ca, 1 / cb, -1 / ra]); dout = np.array([1 / ca, 1 / cb, -1 / rb])
    return ((dout[:, None] * Sc) / din[None, :])[:, [0, 2, 1]] 

def orifice_RT(pt_in):
    prob, res, est = solve_orifice(pt_in)
    if not res.converged or est[ES_M, 1] >= 1.0:
        return None
    p1, p2 = est[ES_P, 0], est[ES_P, 2]
    S_fns = perturbation_response(prob, res.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0].real
    T_ref = (np.linalg.inv(char_to_dq(est[ES_RHO, 2], est[ES_C, 2]))
             @ jump_tm(r_borda, prim(est, 1), prim(est, 2), AT, A2)
             @ jump_tm(r_isen, prim(est, 0), prim(est, 1), A1, AT)
             @ char_to_dq(est[ES_RHO, 0], est[ES_C, 0])).real
    S_ref = char_tm_to_riemann_sm(T_ref, est, 0, 2)
    return est[ES_M, 1], (S_fns[0, 0], S_fns[1, 0] * (p2 / p1)), (S_ref[0, 0], S_ref[1, 0] * (p2 / p1))

rows = [r for r in (orifice_RT(pt) for pt in pt_grid) if r]
MT = np.array([r[0] for r in rows])
Rp_fns = np.array([r[1][0] for r in rows]); Tp_fns = np.array([r[1][1] for r in rows])
Rp_ref = np.array([r[2][0] for r in rows]); Tp_ref = np.array([r[2][1] for r in rows])

BLACK, RED = "#111827", COLORWAY[4]

LEGEND_POSITION = dict(
    orientation="h",
    yanchor="bottom", y=1.01,
    xanchor="center", x=0.5,
    borderwidth=0
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=MT, y=Rp_ref, mode="lines", name=r"$R^+\ \text{(analytic)}$", line=dict(color=BLACK)))
fig.add_trace(go.Scatter(x=MT, y=Tp_ref, mode="lines", name=r"$T^+\ \text{(analytic)}$", line=dict(color=RED)))
fig.add_trace(go.Scatter(x=MT, y=Rp_fns, mode="markers", name=r"$R^+\ \text{(Nefes)}$", marker=dict(color=BLACK, symbol="circle-open", size=7)))
fig.add_trace(go.Scatter(x=MT, y=Tp_fns, mode="markers", name=r"$T^+\ \text{(Nefes)}$", marker=dict(color=RED, symbol="circle-open", size=7)))
fig.update_layout(
    title="Orifice-plate reflectivity and transmissivity (De Domenico Fig. 12)",
    xaxis_title=r"$\text{throat Mach } M_T$", yaxis_title=r"$R^+,\ T^+$",
    xaxis_range=[0, 1], yaxis_range=[0, 1],
    height=460, width=720,
    legend=LEGEND_POSITION
)
             
fig.show()
dev = max(np.max(np.abs(Rp_fns - Rp_ref)), np.max(np.abs(Tp_fns - Tp_ref)))
print(f"max |Nefes - independent analytic composition| over the sweep = {dev:.2e}")

## 5. Open-jet termination — reflected direct & indirect noise (Fig. 16)

Configuration (ii) of the rig (their Fig. 8): the duct is *terminated* by a converging
shape discharging as a jet into the open ($A_2\to\infty$) — a **different network** from
the in-duct orifice of §1–§4. The flow contracts isentropically to the vena contracta
$A_j=\Gamma A_T$; the non-isentropic case then dumps through a lossy jet into the open,
the isentropic case is a convergent nozzle terminating at $A_j$. Forcing is one-sided
from upstream (the heating grid); the open jet is the downstream acoustic closure.

The downstream is quiescent ($A_2\to\infty$), so the pairwise scattering matrix across
the termination does not exist — the **upstream-port** reflections are read from the
multiport descriptor, giving the reflected *direct* and *indirect* noise (their Eq. 24):

$$R = \frac{P^-_1}{P^+_1}, \qquad S_R = \frac{P^-_{1,s}}{\sigma} = -\frac{\rho_1}{c_1}\,\frac{g_1}{h_1}.$$

Non-isentropic predictions (solid): sharp orifice $\Gamma=\Gamma_{OP}$ (their Eq. 26) and
converging nozzle $\Gamma=0.9$; isentropic predictions (dashed). Reproducing their §4.3.1:
$R$ changes sign at low Mach (the cancellation of their Fig. 7), and the isentropic model
**over-predicts** $R$ while **under-predicting** $|S_R|$ — the losses matter.

In [ ]:
A2_OPEN = 200.0 * A1  # converging shape discharges into the open environment (A2 -> inf)


def gamma_op(MT):  # sharp-edged orifice vena-contracta factor (their Eq. 26)
    return 1.0 / (1.0 + (0.5 * (1.0 - AT / A1)) ** 0.5) + 0.13 * MT * MT


def _build_term(pt_in, Gamma, lossy):
    """Compiled config-(ii) problem + converged subsonic mean state (or None)."""
    Aj = Gamma * AT
    if lossy:  # converging shape, then a Borda jet dump into the open
        net = [cat.total_pressure_inlet(pt_in, 300.0), cat.isentropic_area_change(),
               cat.sudden_area_change(), cat.pressure_outlet(101325.0, 300.0)]
        edges = [(0, 1, A1), (1, 2, Aj), (2, 3, A2_OPEN)]
    else:  # isentropic convergent nozzle terminating at Aj
        net = [cat.total_pressure_inlet(pt_in, 300.0), cat.isentropic_area_change(),
               cat.pressure_outlet(101325.0, 300.0)]
        edges = [(0, 1, A1), (1, 2, Aj)]
    prob = build_problem(CFG, net, edges, 1.0, 101325.0, CP * 300.0)
    sol = solve(prob)
    if not sol.converged:
        return None
    est = states_table(prob, sol.x)
    return (prob, sol, est) if est[ES_M, 1] < 1.0 else None


def termination_point(pt_in, family, lossy):
    """(M_T, R, S_R) for an open-jet termination; Gamma_OP fixed-point for the orifice."""
    G = 0.9 if family == "nozzle" else 0.62
    built = _build_term(pt_in, G, lossy)
    if family == "orifice":
        for _ in range(8):  # Gamma_OP depends on M_T (their Eq. 26)
            if built is None:
                return None
            Gn = gamma_op(built[2][ES_M, 1])
            if abs(Gn - G) < 1e-6:
                break
            G, built = Gn, _build_term(pt_in, Gn, lossy)
    if built is None:
        return None
    prob, sol, est = built
    # quiescent open termination -> read the upstream-port reflections from the multiport matrix
    Smp = perturbation_response(prob, sol.x, np.array([0.0]), excite=FULL).multiport_scattering_matrix()[0].real
    return est[ES_M, 1], Smp[0, 0], -(est[ES_RHO, 0] / est[ES_C, 0]) * Smp[0, 2]


def termination_sweep(family, lossy):
    # cluster toward the back pressure to resolve the steep low-Mach reflection rise
    grid = 101330.0 + (240000.0 - 101330.0) * np.linspace(0.0, 1.0, 90) ** 2
    rows = [r for r in (termination_point(pt, family, lossy) for pt in grid) if r]
    rows.sort(key=lambda row: row[0])
    return np.array(rows) if rows else np.empty((0, 3))


cases = [("orifice", True, COLORWAY[4], "solid", r"orifice $\Gamma_{OP}$ (non-isen.)"),
         ("nozzle", True, COLORWAY[0], "solid", r"nozzle $\Gamma=0.9$ (non-isen.)"),
         ("orifice", False, COLORWAY[4], "dash", "orifice (isentropic)"),
         ("nozzle", False, COLORWAY[0], "dash", "nozzle (isentropic)")]
results = {}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=(r"Reflected direct noise $R$", r"Entropy → reflected sound $S_R$ (indirect)"))
for family, lossy, color, dashv, label in cases:
    d = termination_sweep(family, lossy)
    results[(family, lossy)] = d
    if len(d) == 0:
        continue
    fig.add_trace(go.Scatter(x=d[:, 0], y=d[:, 1], name=label, legendgroup=label,
                             line=dict(color=color, dash=dashv)), row=1, col=1)
    fig.add_trace(go.Scatter(x=d[:, 0], y=d[:, 2], legendgroup=label, showlegend=False,
                             line=dict(color=color, dash=dashv)), row=1, col=2)
fig.add_hline(y=0.0, line=dict(dash="dot", color="#9aa5b1"), row=1, col=1)

for c in (1, 2):
    fig.update_xaxes(title_text=r"$M_T$", row=1, col=c)

fig.update_layout(
    title=dict(text="Open-jet termination: reflected direct & indirect noise (De Domenico et. al. Fig. 16)",
               automargin=True),
    height=460, width=980,
    legend=dict(orientation="h", yanchor="top", y=0.975,
                xanchor="center", x=0.5, xref="container", yref="container",
                borderwidth=0, bgcolor="rgba(0,0,0,0)"),
)
fig.show()

# physical checkpoints: the low-Mach sign change in R, and the isentropic over/under-prediction
no, iso = results[("nozzle", True)], results[("nozzle", False)]
sign = np.where(np.diff(np.sign(no[:, 1])))[0]
cross = no[sign[0], 0] if len(sign) else float("nan")
i, j = np.argmin(np.abs(no[:, 0] - 0.8)), np.argmin(np.abs(iso[:, 0] - 0.8))
print(f"R sign change (low-frequency cancellation) near M_T = {cross:.2f}")
print(f"nozzle at M_T=0.8:  R = {no[i,1]:+.2f} (non-isen.) vs {iso[j,1]:+.2f} (isen.)  -> isentropic over-predicts R")
print(f"                    |S_R| = {abs(no[i,2])*1e3:.1f} vs {abs(iso[j,2])*1e3:.1f}  (x1e-3) -> isentropic under-predicts")

## 6. The isentropic-nozzle limit ($\beta = 1$)

In the compact approximation, a **fully isentropic nozzle with
$A_1=A_2$** has acoustic transmission $T^+ = 1$ and reflection $R^+ = 0$ (mass, total
enthalpy and entropy simply pass through).  At the Cambridge contraction
($A_1/A_T\approx 42$) the isentropic throat is *always choked* ($M_T=1$) — the v1
subsonic limit, and exactly the $M_T = 1$ discontinuity in their Fig. 6 — so we
confirm the limit on a milder, genuinely-subsonic geometry.

In [ ]:
# moderate isentropic nozzle: contraction A1->AT then isentropic expansion AT->A2 (= A1)
a1 = a2 = 0.020; at = 0.010
net = [cat.mass_flow_inlet(2.0, 300.0), cat.isentropic_area_change(),
       cat.isentropic_area_change(), cat.pressure_outlet(101325.0, 300.0)]
prob = build_problem(CFG, net, [(0, 1, a1), (1, 2, at), (2, 3, a2)], 3.0, 101325.0, CP * 300.0)
sol = solve(prob); est = states_table(prob, sol.x)
S = perturbation_response(prob, sol.x, np.array([0.0]), excite=FULL).scattering_matrix(0, 2, basis="riemann")[0]
print(f"isentropic nozzle, A1 = A2, M_T = {est[ES_M,1]:.3f},  p_t loss = {est[ES_PT,0]-est[ES_PT,2]:.2e} Pa")
print(f"  R+ = {S[0,0].real:+.6f}")
print(f"  T+ = {(S[1,0]*(est[ES_P,2]/est[ES_P,0])).real:+.6f}")

## Export for Nemo

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in Nemo.

In [ ]:
from nefes.shell import Network, Solution

network = Network(CFG, p_ref=101325.0, mdot_ref=3.0, h_ref=CP * 300.0,
                  nodes=net, edges=[(0, 1, a1), (1, 2, at), (2, 3, a2)])
sol = Solution(network, prob, sol)  # rebind `sol` to the shell Solution (old `sol` was the SolveResult)

import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "entropy_generator.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)